# Data Extractor — plataforma de datos de fútbol

Pipeline de extracción para construir la base de datos de partidos (clubes + selecciones) del proyecto. Objetivo: el mayor número de variables útiles por partido, con una extracción lo más robusta posible.

**Sobre "perfecta"**: ningún scraper sobre sitios de terceros lo es — pueden bloquear, cambiar el HTML o no tener datos para épocas antiguas. En vez de perseguir cero fallos, este notebook está pensado para que un fallo puntual (una liga, una temporada, un stat_type) no tire el resto: cada extracción pasa por un wrapper con reintentos, se guarda en cuanto está lista, y queda registrada en un manifiesto para saber exactamente qué se consiguió y qué no.

**Fuente principal: `soccerdata`.** Es la librería más madura de las dos que hemos probado (activamente mantenida, cache local, mensajes de error claros). Cubre bien: las 5 grandes ligas de clubes (Premier League, LaLiga, Serie A, Bundesliga, Ligue 1) y, a nivel de selecciones, **Mundial, Eurocopa y Mundial femenino** — confirmado en la exploración anterior que FBref tiene calendario desde 1930 (Mundial), 2000 (Eurocopa) y 1991 (Mundial femenino), y que `read_schedule()`/`read_leagues()` funcionan de verdad.

**Huecos de soccerdata**: Copa América, Copa África, Nations League, eliminatorias y amistosos no están en su `LEAGUE_DICT`. Para "todos los partidos de selecciones desde 1872" seguimos teniendo `data/results.csv`, que ya cubre eso a nivel básico.

**Novedad confirmada sobre `cesc-football-scraper`** (apéndice al final): su catálogo de ESPN (`list_competitions_available`, 151 competiciones) **sí incluye Copa América, Copa África, Nations League y eliminatorias por confederación** — justo lo que le falta a soccerdata. Pero el paso siguiente (`extract_teams`/`extract_results`, pensado para ligas de clubes) sigue dando timeouts de Selenium en torneos de selecciones, así que de momento no es fiable para extraer partidos, solo para saber qué competiciones existen.

## 0. Instalación e imports

In [18]:
!py -m pip install -q soccerdata

In [19]:
import logging
import time
from pathlib import Path

import pandas as pd
import soccerdata as sd

pd.set_option("display.max_columns", None)

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger("data_extractor")

## 1. Utilidades: reintentos, guardado y manifiesto

`safe_call` envuelve cualquier llamada a soccerdata con reintentos y, si falla del todo, devuelve `None` en vez de tirar abajo el notebook. `save` escribe el resultado a disco inmediatamente (no esperamos a tener todo para guardar nada) y anota el resultado en `manifest`, que al final nos dice qué se extrajo y qué no.

In [20]:
RAW_DIR = Path("../data/raw/soccerdata")
RAW_DIR.mkdir(parents=True, exist_ok=True)

manifest = []


def safe_call(fn, *args, retries: int = 2, wait: float = 5.0, **kwargs):
    """Llama a fn(*args, **kwargs) con reintentos. Devuelve None si todos fallan."""
    last_error = None
    for attempt in range(1, retries + 2):
        try:
            return fn(*args, **kwargs)
        except Exception as exc:  # noqa: BLE001 - queremos capturar cualquier fallo de scraping
            last_error = exc
            logger.warning("Intento %d/%d fallido (%s): %s", attempt, retries + 1, fn, exc)
            if attempt <= retries:
                time.sleep(wait)
    logger.error("Se agotaron los reintentos para %s: %s", fn, last_error)
    return None


def save(df, name: str):
    """Guarda df en RAW_DIR/name.csv y registra el resultado en manifest."""
    if df is None or len(df) == 0:
        manifest.append({"dataset": name, "filas": 0, "columnas": 0, "estado": "FALLIDO/VACIO"})
        return None
    path = RAW_DIR / f"{name}.csv"
    df.to_csv(path)
    manifest.append(
        {"dataset": name, "filas": len(df), "columnas": df.shape[1], "estado": "OK", "path": str(path)}
    )
    return path

## 2. Configuración de competiciones

Empezamos con rangos de temporadas moderados para validar que todo el pipeline corre limpio de principio a fin. Una vez confirmado, se amplía `SEASONS_CLUB` hacia atrás sin tocar el resto del notebook.

Los años de los torneos internacionales están puestos a mano (son datos históricos conocidos, no hay forma fiable de "descubrirlos" vía la API de soccerdata sin haberlos fijado antes).

In [21]:
# Ligas de clubes: usamos el agregado "Big 5" porque soccerdata lo recomienda
# (una sola pasada en vez de instanciar 5 readers distintos).
CLUB_LEAGUES = "Big 5 European Leagues Combined"
SEASONS_CLUB = list(range(2018, 2027))  # amplía hacia atrás cuando confirmes que va bien

# Torneos de selecciones soportados por el LEAGUE_DICT de soccerdata (vía FBref).
WORLD_CUP_YEARS = [1930, 1934, 1938, 1950, 1954, 1958, 1962, 1966, 1970, 1974,
                    1978, 1982, 1986, 1990, 1994, 1998, 2002, 2006, 2010, 2014,
                    2018, 2022, 2026]
EURO_YEARS = [1960, 1964, 1968, 1972, 1976, 1980, 1984, 1988, 1992, 1996,
              2000, 2004, 2008, 2012, 2016, 2020, 2024]
WOMENS_WORLD_CUP_YEARS = [1991, 1995, 1999, 2003, 2007, 2011, 2015, 2019, 2023]

# Advertencia: read_team_match_stats necesita que exista la página de "Match Report"
# de cada partido. Para ediciones muy antiguas puede no existir -> safe_call
# simplemente devolverá None para esos casos y seguirá con el resto.
INTL_STAT_TYPES = ["shooting", "keeper", "misc"]
CLUB_STAT_TYPES = ["shooting", "keeper", "misc"]

## 3. Ligas de clubes (FBref)

`schedule` da el partido a partido (fecha, equipos, resultado, xG, asistencia, árbitro, sede). `shooting` / `keeper` / `misc` dan estadísticas avanzadas por equipo y partido (tiros, paradas, faltas, córners, tarjetas...). Cada `stat_type` hace scraping equipo por equipo, así que puede tardar bastante — por eso pasa por `safe_call` y se guarda cada uno según va saliendo.

In [22]:
fbref_clubs = sd.FBref(leagues=CLUB_LEAGUES, seasons=SEASONS_CLUB)

df_club_schedule = safe_call(fbref_clubs.read_team_match_stats, stat_type="schedule")
save(df_club_schedule, "fbref_club_schedule")
df_club_schedule.head() if df_club_schedule is not None else None

[07/03/26 14:46:08] INFO     Saving cached data to C:\Users\daniel.cejas\soccerdata\data\FBref       ]8;id=11222039;file://c:\Users\daniel.cejas\AppData\Local\Programs\Python\Python314\Lib\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=11222040;file://c:\Users\daniel.cejas\AppData\Local\Programs\Python\Python314\Lib\site-packages\soccerdata\_common.py#250\250]8;;\

[07/03/26 14:46:28] WARNING  Intento 1/3 fallido (<bound method FBref.read_team_match_stats of      ]8;id=11222047;file://C:\Users\daniel.cejas\AppData\Local\Temp\ipykernel_25968\146687142.py\146687142.py]8;;\:]8;id=11222048;file://C:\Users\daniel.cejas\AppData\Local\Temp\ipykernel_25968\146687142.py#15\15]8;;\
                             <soccerdata.fbref.FBref object at 0x000002198ECED950>>): '2627'                       

[07/03/26 14:46:33] WARNING  Intento 2/3 fallido (<bound method FBref.read_team_match_stats of      ]8;id=11222053;file://C:\Users\daniel.cejas\AppData\Local\Temp\ipykernel_25968\146687142.py\146687142.py]8;;\:]8;id=11222054;file://C:\Users\daniel.cejas\AppData\Local\Temp\ipykernel_25968\146687142.py#15\15]8;;\
                             <soccerdata.fbref.FBref object at 0x000002198ECED950>>): '2627'                       

[07/03/26 14:46:38] WARNING  Intento 3/3 fallido (<bound method FBref.read_team_match_stats of      ]8;id=11222059;file://C:\Users\daniel.cejas\AppData\Local\Temp\ipykernel_25968\146687142.py\146687142.py]8;;\:]8;id=11222060;file://C:\Users\daniel.cejas\AppData\Local\Temp\ipykernel_25968\146687142.py#15\15]8;;\
                             <soccerdata.fbref.FBref object at 0x000002198ECED950>>): '2627'                       

                    ERROR    Se agotaron los reintentos para <bound method                          ]8;id=11222066;file://C:\Users\daniel.cejas\AppData\Local\Temp\ipykernel_25968\146687142.py\146687142.py]8;;\:]8;id=11222067;file://C:\Users\daniel.cejas\AppData\Local\Temp\ipykernel_25968\146687142.py#18\18]8;;\
                             FBref.read_team_match_stats of <soccerdata.fbref.FBref object at                      
                             0x000002198ECED950>>: '2627'                                                          

In [23]:
for stat_type in CLUB_STAT_TYPES:
    df = safe_call(fbref_clubs.read_team_match_stats, stat_type=stat_type)
    save(df, f"fbref_club_{stat_type}")

pd.DataFrame(manifest)

[07/03/26 14:46:54] WARNING  Intento 1/3 fallido (<bound method FBref.read_team_match_stats of      ]8;id=11222072;file://C:\Users\daniel.cejas\AppData\Local\Temp\ipykernel_25968\146687142.py\146687142.py]8;;\:]8;id=11222073;file://C:\Users\daniel.cejas\AppData\Local\Temp\ipykernel_25968\146687142.py#15\15]8;;\
                             <soccerdata.fbref.FBref object at 0x000002198ECED950>>): '2627'                       

[07/03/26 14:46:59] WARNING  Intento 2/3 fallido (<bound method FBref.read_team_match_stats of      ]8;id=11222078;file://C:\Users\daniel.cejas\AppData\Local\Temp\ipykernel_25968\146687142.py\146687142.py]8;;\:]8;id=11222079;file://C:\Users\daniel.cejas\AppData\Local\Temp\ipykernel_25968\146687142.py#15\15]8;;\
                             <soccerdata.fbref.FBref object at 0x000002198ECED950>>): '2627'                       

[07/03/26 14:47:04] WARNING  Intento 3/3 fallido (<bound method FBref.read_team_match_stats of      ]8;id=11222084;file://C:\Users\daniel.cejas\AppData\Local\Temp\ipykernel_25968\146687142.py\146687142.py]8;;\:]8;id=11222085;file://C:\Users\daniel.cejas\AppData\Local\Temp\ipykernel_25968\146687142.py#15\15]8;;\
                             <soccerdata.fbref.FBref object at 0x000002198ECED950>>): '2627'                       

                    ERROR    Se agotaron los reintentos para <bound method                          ]8;id=11222090;file://C:\Users\daniel.cejas\AppData\Local\Temp\ipykernel_25968\146687142.py\146687142.py]8;;\:]8;id=11222091;file://C:\Users\daniel.cejas\AppData\Local\Temp\ipykernel_25968\146687142.py#18\18]8;;\
                             FBref.read_team_match_stats of <soccerdata.fbref.FBref object at                      
                             0x000002198ECED950>>: '2627'                                                          

                    WARNING  Intento 1/3 fallido (<bound method FBref.read_team_match_stats of      ]8;id=11222096;file://C:\Users\daniel.cejas\AppData\Local\Temp\ipykernel_25968\146687142.py\146687142.py]8;;\:]8;id=11222097;file://C:\Users\daniel.cejas\AppData\Local\Temp\ipykernel_25968\146687142.py#15\15]8;;\
                             <soccerdata.fbref.FBref object at 0x000002198ECED950>>): '2627'                       

...

                    WARNING  Intento 1/3 fallido (<bound method FBref.read_team_match_stats of      ]8;id=11222120;file://C:\Users\daniel.cejas\AppData\Local\Temp\ipykernel_25968\146687142.py\146687142.py]8;;\:]8;id=11222121;file://C:\Users\daniel.cejas\AppData\Local\Temp\ipykernel_25968\146687142.py#15\15]8;;\
                             <soccerdata.fbref.FBref object at 0x000002198ECED950>>): '2627'                       

[07/03/26 14:47:20] WARNING  Intento 2/3 fallido (<bound method FBref.read_team_match_stats of      ]8;id=11222126;file://C:\Users\daniel.cejas\AppData\Local\Temp\ipykernel_25968\146687142.py\146687142.py]8;;\:]8;id=11222127;file://C:\Users\daniel.cejas\AppData\Local\Temp\ipykernel_25968\146687142.py#15\15]8;;\
                             <soccerdata.fbref.FBref object at 0x000002198ECED950>>): '2627'                       

[07/03/26 14:47:25] WARNING  Intento 3/3 fallido (<bound method FBref.read_team_match_stats of      ]8;id=11222132;file://C:\Users\daniel.cejas\AppData\Local\Temp\ipykernel_25968\146687142.py\146687142.py]8;;\:]8;id=11222133;file://C:\Users\daniel.cejas\AppData\Local\Temp\ipykernel_25968\146687142.py#15\15]8;;\
                             <soccerdata.fbref.FBref object at 0x000002198ECED950>>): '2627'                       

                    ERROR    Se agotaron los reintentos para <bound method                          ]8;id=11222138;file://C:\Users\daniel.cejas\AppData\Local\Temp\ipykernel_25968\146687142.py\146687142.py]8;;\:]8;id=11222139;file://C:\Users\daniel.cejas\AppData\Local\Temp\ipykernel_25968\146687142.py#18\18]8;;\
                             FBref.read_team_match_stats of <soccerdata.fbref.FBref object at                      
                             0x000002198ECED950>>: '2627'                                                          

,dataset,filas,columnas,estado
0,fbref_club_schedule,0,0,FALLIDO/VACIO
1,fbref_club_shooting,0,0,FALLIDO/VACIO
2,fbref_club_keeper,0,0,FALLIDO/VACIO
3,fbref_club_misc,0,0,FALLIDO/VACIO


## 4. Torneos de selecciones (FBref)

Mismo patrón que las ligas de clubes, pero pasando los años reales de cada torneo (ver sección 2). Primero el calendario completo, luego las estadísticas avanzadas — estas últimas es muy probable que fallen para ediciones anteriores a los 2000 (FBref no tiene "Match Report" para partidos tan antiguos), y está bien: `safe_call` lo registrará como fallido y seguimos.

In [ ]:
fbref_world_cup = sd.FBref(leagues="INT-World Cup", seasons=WORLD_CUP_YEARS)
fbref_euro = sd.FBref(leagues="INT-European Championship", seasons=EURO_YEARS)
fbref_wwc = sd.FBref(leagues="INT-Women's World Cup", seasons=WOMENS_WORLD_CUP_YEARS)

intl_readers = {
    "world_cup": fbref_world_cup,
    "euro": fbref_euro,
    "womens_world_cup": fbref_wwc,
}

for comp_name, reader in intl_readers.items():
    df_schedule = safe_call(reader.read_schedule)
    save(df_schedule, f"fbref_{comp_name}_schedule")

pd.DataFrame(manifest)

In [ ]:
for comp_name, reader in intl_readers.items():
    for stat_type in INTL_STAT_TYPES:
        df = safe_call(reader.read_team_match_stats, stat_type=stat_type)
        save(df, f"fbref_{comp_name}_{stat_type}")

pd.DataFrame(manifest)

## 5. Fuerza de equipos de clubes (ClubElo)

Rating Elo diario por club, útil como variable de "fuerza del equipo" para el modelo. Ojo: en la prueba anterior `ClubElo.read_by_date()` dio timeout de red en esta máquina (`http://api.clubelo.com` puede estar bloqueado por firewall/proxy corporativo) — por eso esta sección es opcional y no bloqueante: si falla, el resto del notebook sigue funcionando igual.

Sacamos el historial de Elo por equipo (`read_team_history`) para los equipos que ya aparecen en el calendario de clubes extraído arriba.

In [ ]:
elo = sd.ClubElo()

elo_history = []
if df_club_schedule is not None:
    teams = sorted(set(df_club_schedule["home_team"]) | set(df_club_schedule["away_team"]))
    for team in teams:
        df_team_elo = safe_call(elo.read_team_history, team, retries=1, wait=3)
        if df_team_elo is not None:
            df_team_elo["team"] = team
            elo_history.append(df_team_elo)

df_elo = pd.concat(elo_history) if elo_history else None
save(df_elo, "clubelo_team_history")

pd.DataFrame(manifest).tail(5)

## 6. Odds y estadísticas adicionales de clubes (MatchHistory / football-data.co.uk)

Otra fuente dentro de soccerdata, solo para las 5 grandes ligas: cuotas de casas de apuestas, goles al descanso, tiros, córners, tarjetas y árbitro — variables que FBref no siempre tiene, sobre todo en temporadas más antiguas.

In [ ]:
match_history = sd.MatchHistory(
    leagues=["ENG-Premier League", "ESP-La Liga", "ITA-Serie A", "GER-Bundesliga", "FRA-Ligue 1"],
    seasons=SEASONS_CLUB,
)
df_match_history = safe_call(match_history.read_games)
save(df_match_history, "match_history_club_odds")
df_match_history.head() if df_match_history is not None else None

## 7. Resumen de la extracción

Esto es lo importante para saber si "fue perfecta": qué datasets se consiguieron, cuántas filas tiene cada uno, y qué falló.

In [ ]:
df_manifest = pd.DataFrame(manifest)
df_manifest.to_csv(RAW_DIR / "_manifest.csv", index=False)
df_manifest

## 8. Diccionario de variables (por dataset)

| Dataset | Fuente | Cobertura | Variables clave |
|---|---|---|---|
| `fbref_club_schedule` | FBref | Big 5, desde `SEASONS_CLUB` | fecha, equipos, resultado, xG, asistencia, árbitro, sede, jornada |
| `fbref_club_shooting` / `_keeper` / `_misc` | FBref | Big 5, desde `SEASONS_CLUB` | tiros, tiros a puerta, paradas, faltas, córners, tarjetas |
| `fbref_world_cup_schedule` | FBref | 1930–2026 | fecha, equipos, resultado, ronda, sede, árbitro, asistencia |
| `fbref_euro_schedule` | FBref | 1960–2024 | idem |
| `fbref_womens_world_cup_schedule` | FBref | 1991–2023 | idem |
| `fbref_{world_cup,euro,womens_world_cup}_{shooting,keeper,misc}` | FBref | variable (fiable ~2000 en adelante) | mismas variables avanzadas que en clubes |
| `clubelo_team_history` | ClubElo | histórico por club (best-effort) | rating Elo, rango de fechas de validez |
| `match_history_club_odds` | football-data.co.uk | Big 5, desde `SEASONS_CLUB` | cuotas 1X2/over-under, goles al descanso, tiros, córners, tarjetas, árbitro |

Todo queda en `data/raw/soccerdata/*.csv`, un archivo por dataset, más `_manifest.csv` con el resumen.

## 9. Huecos conocidos y próximos pasos

- **No cubierto por este pipeline**: Copa América, Copa África, Nations League, eliminatorias, amistosos. Para eso seguimos dependiendo de `data/results.csv` (1872–hoy, nivel básico) — cruzar por fecha+equipos si se necesitan enriquecer esos partidos.
- **El catálogo de ESPN sí tiene esas competiciones** (ver apéndice: 151 competiciones, incluye `Copa América`, `Africa Cup of Nations`, `UEFA Nations League`, `FIFA World Cup Qualifying - CONMEBOL/CAF/AFC/...`). El problema no es que no existan los datos, es que el método de extracción de partidos de `cesc-football-scraper` (pensado para ligas de clubes) no funciona todavía de forma fiable sobre ellas.
- **Estadísticas avanzadas en torneos antiguos**: revisa el manifiesto — si `fbref_world_cup_shooting` etc. salió "FALLIDO/VACIO" para años concretos, es porque FBref no tiene Match Report para esos partidos, no un bug del notebook.
- **ClubElo**: si dio timeout de red, prueba desde otra red o revisa el proxy/firewall — no es necesario para el resto del pipeline.
- **Siguiente paso natural**: una vez el manifiesto salga limpio con `SEASONS_CLUB` reducido, ampliar el rango de temporadas hacia atrás y volver a correr.
- **Ampliar cobertura de selecciones a Copa América/AFCON/Nations League/eliminatorias**: dos caminos — (a) esperar a que `cesc-football-scraper` madure su scraper de equipos/resultados para torneos, o (b) escribir un extractor propio y pequeño que pegue directamente sobre las URLs de ESPN que ya sabemos que existen (`espn.com/soccer/teams/_/league/<slug>`), sin pasar por los métodos rotos de esa librería.

---
## Apéndice A — `cesc-football-scraper`

Se probó como alternativa para ampliar cobertura a más competiciones. Resultado mixto pero útil:

- ✅ **El catálogo de ESPN funciona y confirma cobertura amplia**: `list_competitions_available()` devolvió 151 competiciones, incluyendo Copa América, Copa África, Nations League y eliminatorias por confederación — justo lo que le falta a soccerdata.
- ⚠️ `selenium`/`webdriver_manager` son dependencia *opcional* mal comunicada (`pip install cesc-football-scraper` a secas falla con `ModuleNotFoundError`; hace falta `[selenium]`).
- ❌ `client.espn.team.extract_results(...)` sigue dando `TimeoutException`/`WebDriverException` con competiciones reales de selecciones (`"FIFA World Cup"`) — la página de equipos existe y tiene los datos (comprobado con una petición directa), así que es fragilidad del método/driver de la librería para este tipo de competición, no ausencia de datos. `extract_teams`/`extract_results` están pensados para el patrón de ligas de clubes (roster de equipos permanente) y no encajan bien con el formato de torneo.
- Repo de 2 estrellas, un mantenedor, sin releases: alto riesgo de romperse sin aviso.

**Conclusión**: útil para *descubrir* qué competiciones de selecciones existen en ESPN, no todavía para *extraer* sus partidos vía esta librería. Si se necesita de verdad Copa América/AFCON/Nations League con datos ricos, el camino más robusto a corto plazo es un scraper propio y pequeño sobre las URLs de ESPN ya confirmadas, en vez de depender de los métodos inestables de `football_scraper`.

In [ ]:
!py -m pip install -q "cesc-football-scraper[selenium]"

In [ ]:
from football_scraper import FootballClient

client = FootballClient()

In [ ]:
espn_competitions = client.espn.competition.list_competitions_available()
len(espn_competitions), espn_competitions[:30]

(151,
 ['MLS',
  'Mexican Liga BBVA MX',
  'Leagues Cup',
  'Costa Rican Primera Division',
  'Honduran Liga Nacional',
  'USL Championship',
  'USL Cup',
  'Concacaf Central American Cup',
  'NWSL Challenge Cup',
  'Concacaf W Champions Cup',
  "NCAA Women's Soccer",
  'U.S. Open Cup',
  'Mexican Liga de Expansión MX',
  'Concacaf Champions Cup',
  'Guatemalan Liga Nacional',
  'Salvadoran Primera Division',
  'USL League One',
  "NCAA Men's Soccer",
  'NWSL',
  'USL Super League',
  'Northern Super League',
  'UEFA Champions League',
  'UEFA Conference League',
  'English FA Cup',
  'English League Championship',
  'Spanish Copa del Rey',
  'German Cup',
  'Coppa Italia',
  'Coupe de France',
  'Taca de Portugal'])

In [ ]:
keywords = ["world cup", "euro", "copa", "nations", "african", "asian", "concacaf", "qualif"]
international_candidates = [c for c in espn_competitions if any(k in c.lower() for k in keywords)]
international_candidates

['Concacaf Central American Cup',
 'Concacaf W Champions Cup',
 'Concacaf Champions Cup',
 'Spanish Copa del Rey',
 'UEFA Europa League Qualifying',
 'Spanish Copa de la Reina',
 'UEFA Europa League',
 'UEFA Champions League Qualifying',
 'UEFA Conference League Qualifying',
 'FIFA World Cup',
 'FIFA World Cup Qualifying - UEFA',
 'FIFA World Cup Qualifying - Concacaf',
 'FIFA World Cup Qualifying - AFC',
 'FIFA World Cup Qualifying - OFC',
 'FIFA Under-17 World Cup',
 'UEFA European Championship',
 'UEFA Nations League',
 "UEFA Women's Nations League",
 'UEFA European Under-21 Championship Qualifying',
 'Concacaf Gold Cup',
 'Concacaf Nations League',
 'Copa América Femenina',
 'Africa Cup of Nations Qualifying',
 'African Nations Championship',
 'AFC Asian Cup Qualifiers',
 "FIFA Women's World Cup",
 "FIFA Women's World Cup Qualifying - UEFA",
 'FIFA World Cup Qualifying - CONMEBOL',
 'FIFA World Cup Qualifying - CAF',
 'FIFA Under-20 World Cup',
 "FIFA Under-17 Women's World Cup",
 

In [ ]:
# extract_results está pensado para ligas de clubes (roster de equipo fijo por temporada)
# y de momento da timeouts en competiciones de torneo como esta. Se deja para seguir probando.
df_espn_results = client.espn.team.extract_results("Argentina", "FIFA World Cup", "2022")
df_espn_results.head(20)